# 서울시 공원 접근성 시각화 분석

## 분석 개요

본 노트북은 서울시 426개 행정동의 공원 접근성을 8가지 관점에서 시각화하여 분석합니다.

### 분석 항목
1. 거리기준에 따른 행정동별 공원 접근성 점수별 막대그래프
2. WHO기준(9 m²/인) 대비 서울시 평균 접근성 비교 막대그래프
3. 버퍼 거리별 평균 접근성 점수 변화 선그래프
4. 버퍼 거리별 Gini 계수 변화 선그래프
5. 공원 유형별 '공원 수 비율 vs 접근성 기여도' 비교 막대그래프
6. 공원 유형별 접근성 기여 비중 파이차트
7. 행정동별 접근성 상·하위 10개 지역 비교 막대그래프

### 데이터 출처
- 행정동 경계: 브이월드
- 공원 데이터: 서울시 도시공원 현황
- 격자 인구: 국토정보플랫폼 100m 격자 인구수
- 분석 결과: `dong_service_score_multi_buffer_parktype.csv`

In [ ]:
# 필수 라이브러리 임포트
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from pathlib import Path
import warnings
import platform
warnings.filterwarnings('ignore')


plt.rcParams['font.family'] = 'Malgun Gothic'   # 맑은 고딕
plt.rcParams['axes.unicode_minus'] = False      # 마이너스 깨짐 방지


In [ ]:
# 한글 폰트 테스트
fig, ax = plt.subplots(figsize=(8, 4))
ax.text(0.5, 0.5, '한글 폰트 테스트: 서울시 공원 접근성 분석', 
        fontsize=16, ha='center', va='center', fontweight='bold')
ax.text(0.5, 0.3, f'현재 폰트: {plt.rcParams["font.family"]}', 
        fontsize=12, ha='center', va='center')
ax.axis('off')
plt.tight_layout()
plt.show()

print("✅ 한글이 정상적으로 표시되면 폰트 설정이 완료된 것입니다.")

In [ ]:
# 데이터 경로 설정
base_path = Path('../s1_data/s2_processed_data')

# 다중 버퍼 분석 결과 로드
df_multi = pd.read_csv(base_path / 'dong_service_score_multi_buffer_parktype.csv', encoding='utf-8-sig')
print(f"다중 버퍼 분석 데이터: {len(df_multi):,}개 레코드")
print(f"버퍼 거리: {sorted(df_multi['buffer_distance'].unique())}")
print(f"공원 유형 수: {len(df_multi['park_types'].unique())}개")

# 공원 데이터 로드
gdf_park = gpd.read_file(base_path / 'seoul_parks.geojson')
print(f"\n공원 데이터: {len(gdf_park):,}개 공원")

# 행정동 경계 데이터 로드 (지도용)
gdf_dong = gpd.read_file(base_path / 'seoul_dong_with_pop.geojson')
print(f"행정동 데이터: {len(gdf_dong):,}개 행정동")

# 데이터 미리보기
print("\n데이터 샘플:")
display(df_multi.head())

## 대시보드 평균 서비스 점수 검증 및 계산 로직

### 평균 서비스 점수란?

**평균 서비스 점수 (avg_service_per_capita)**는 서울시 전체 행정동의 1인당 평균 공원 접근성을 나타내는 핵심 지표입니다.

### 계산 방법

```
평균 서비스 점수 = Σ(각 행정동의 avg_service_per_capita) / 행정동 수
```

### 대시보드 검증 (http://127.0.0.1:8050/)

대시보드에서 표시되는 값:
- **분석 지역**: 425개 행정동
- **평균 서비스 점수**: 4.6289 m²/인
- **커버리지**: 97.9%
- **Gini 계수**: 0.768

In [ ]:
# 500m 버퍼 기준 전체 공원 유형 데이터 추출
df_500m_all = df_multi[
    (df_multi['buffer_distance'] == 500) & 
    (df_multi['park_types'] == '전체')
].copy()

# 평균 서비스 점수 계산
avg_service_score = df_500m_all['avg_service_per_capita'].mean()
avg_coverage = df_500m_all['coverage_ratio'].mean()
total_dongs = len(df_500m_all)

# Gini 계수 계산
def calculate_gini(values):
    """Gini 계수 계산"""
    values = np.array(values)
    values = values[values > 0]
    if len(values) == 0:
        return 0
    values = np.sort(values)
    n = len(values)
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * values)) / (n * np.sum(values)) - (n + 1) / n

gini_coef = calculate_gini(df_500m_all['avg_service_per_capita'].values)

# 결과 출력
print("="*80)
print("대시보드 값 검증 (500m 버퍼, 전체 공원 유형)")
print("="*80)
print(f"\n✅ 분석 지역 수: {total_dongs}개 행정동")
print(f"✅ 평균 서비스 점수: {avg_service_score:.4f} m²/인")
print(f"✅ 평균 커버리지: {avg_coverage:.2f}%")
print(f"✅ Gini 계수: {gini_coef:.4f}")

# 대시보드 값과 비교
print(f"\n{'='*80}")
print("대시보드 표시 값과 일치 확인:")
print(f"{'='*80}")
print(f"대시보드: 4.6289 m²/인")
print(f"계산 결과: {avg_service_score:.4f} m²/인")
print(f"일치 여부: {'✅ 일치' if abs(avg_service_score - 4.6289) < 0.0001 else '❌ 불일치'}")

# 통계 요약
print(f"\n{'='*80}")
print("서비스 점수 분포 통계:")
print(f"{'='*80}")
print(f"최소값: {df_500m_all['avg_service_per_capita'].min():.4f} m²/인")
print(f"25%: {df_500m_all['avg_service_per_capita'].quantile(0.25):.4f} m²/인")
print(f"중앙값: {df_500m_all['avg_service_per_capita'].median():.4f} m²/인")
print(f"75%: {df_500m_all['avg_service_per_capita'].quantile(0.75):.4f} m²/인")
print(f"최대값: {df_500m_all['avg_service_per_capita'].max():.4f} m²/인")
print(f"표준편차: {df_500m_all['avg_service_per_capita'].std():.4f} m²/인")

# WHO 기준 충족률
who_compliant = df_500m_all[df_500m_all['avg_service_per_capita'] >= 9]
print(f"\nWHO 기준(9 m²/인) 충족 행정동: {len(who_compliant)}개 / {total_dongs}개")
print(f"충족률: {len(who_compliant)/total_dongs*100:.1f}%")

## 1. 거리기준에 따른 행정동별 공원 접근성 점수 막대그래프

500m 버퍼 기준으로 행정동별 접근성 점수 상위 20개 지역을 시각화합니다.

In [ ]:
# 500m 기준 전체 공원 유형 데이터 필터링
df_500m = df_multi[
    (df_multi['buffer_distance'] == 500) & 
    (df_multi['park_types'] == '전체')
].copy()

print(f"분석 대상: {len(df_500m)}개 행정동")
print(f"평균 서비스 점수: {df_500m['avg_service_per_capita'].mean():.4f} m²/인")
print(f"평균 커버리지: {df_500m['coverage_ratio'].mean():.2f}%")
print()

# 상위 20개 행정동 추출
top20_dong = df_500m.nlargest(20, 'avg_service_per_capita').sort_values('avg_service_per_capita')

# 시각화
fig, ax = plt.subplots(figsize=(12, 10))

bars = ax.barh(
    range(len(top20_dong)),
    top20_dong['avg_service_per_capita'],
    color=sns.color_palette('viridis', len(top20_dong))
)

ax.set_yticks(range(len(top20_dong)))
ax.set_yticklabels([f"{row['ADM_NM']} ({row['SIGUNGU_NM']})" 
                    for _, row in top20_dong.iterrows()],
                   fontsize=10)

ax.set_xlabel('공원 접근성 점수 (m²/인)', fontsize=12, fontweight='bold')
ax.set_title('거리기준(500m) 공원 접근성 상위 20개 행정동', 
             fontsize=16, fontweight='bold', pad=20)

# WHO 기준선 표시
ax.axvline(x=9, color='red', linestyle='--', linewidth=2, label='WHO 권장 기준 (9 m²/인)')
# 서울시 평균선 표시
seoul_avg = df_500m['avg_service_per_capita'].mean()
ax.axvline(x=seoul_avg, color='blue', linestyle=':', linewidth=2, 
           label=f'서울시 평균 ({seoul_avg:.2f} m²/인)')
ax.legend(fontsize=11, loc='lower right')

# 값 레이블 추가
for i, (idx, row) in enumerate(top20_dong.iterrows()):
    ax.text(row['avg_service_per_capita'] + 0.5, i, 
            f"{row['avg_service_per_capita']:.2f}",
            va='center', fontsize=9)

ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n통계 요약:")
print(f"평균: {df_500m['avg_service_per_capita'].mean():.2f} m²/인")
print(f"최대값: {df_500m['avg_service_per_capita'].max():.2f} m²/인 ({df_500m.loc[df_500m['avg_service_per_capita'].idxmax(), 'ADM_NM']}")
print(f"최소값: {df_500m['avg_service_per_capita'].min():.2f} m²/인 ({df_500m.loc[df_500m['avg_service_per_capita'].idxmin(), 'ADM_NM']})")
print(f"중앙값: {df_500m['avg_service_per_capita'].median():.2f} m²/인")

## 2. WHO 기준(9 m²/인) 대비 시군구별 평균 접근성 비교

시군구별 평균 접근성을 WHO 권장 기준과 비교하여 분석합니다.

In [ ]:
# 시군구별 평균 접근성 계산 (인구 가중 평균)
sigungu_stats = df_500m.groupby('SIGUNGU_NM').apply(
    lambda x: pd.Series({
        'avg_service_per_capita': (
            (x['avg_service_per_capita'] * x['total_population']).sum() / 
            x['total_population'].sum()
        ),
        'total_population': x['total_population'].sum()
    })
).reset_index()

# WHO 기준과의 차이 계산
sigungu_stats['who_diff'] = sigungu_stats['avg_service_per_capita'] - 9
sigungu_stats = sigungu_stats.sort_values('avg_service_per_capita')

# 시각화
fig, ax = plt.subplots(figsize=(14, 10))

colors = ['#d62728' if x < 9 else '#2ca02c' for x in sigungu_stats['avg_service_per_capita']]

bars = ax.barh(
    range(len(sigungu_stats)),
    sigungu_stats['avg_service_per_capita'],
    color=colors,
    alpha=0.7
)

ax.set_yticks(range(len(sigungu_stats)))
ax.set_yticklabels(sigungu_stats['SIGUNGU_NM'], fontsize=10)
ax.set_xlabel('공원 접근성 점수 (m²/인)', fontsize=12, fontweight='bold')
ax.set_title('WHO 기준(9 m²/인) 대비 시군구별 공원 접근성 비교', 
             fontsize=16, fontweight='bold', pad=20)

# WHO 기준선
ax.axvline(x=9, color='red', linestyle='--', linewidth=2.5, label='WHO 권장 기준 (9 m²/인)')
# 서울시 평균선
ax.axvline(x=seoul_avg, color='blue', linestyle=':', linewidth=2, 
           label=f'서울시 평균 ({seoul_avg:.2f} m²/인)')

# 값 레이블
for i, (idx, row) in enumerate(sigungu_stats.iterrows()):
    ax.text(row['avg_service_per_capita'] + 0.3, i, 
            f"{row['avg_service_per_capita']:.2f}",
            va='center', fontsize=9)

ax.legend(fontsize=11, loc='lower right')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# 통계 요약
who_compliant_sigungu = sigungu_stats[sigungu_stats['avg_service_per_capita'] >= 9]
print(f"\nWHO 기준 충족 시군구: {len(who_compliant_sigungu)}개 / {len(sigungu_stats)}개")
print(f"충족률: {len(who_compliant_sigungu)/len(sigungu_stats)*100:.1f}%")
print(f"\n상위 5개 시군구:")
print(sigungu_stats.nlargest(5, 'avg_service_per_capita')[['SIGUNGU_NM', 'avg_service_per_capita']])
print(f"\n하위 5개 시군구:")
print(sigungu_stats.nsmallest(5, 'avg_service_per_capita')[['SIGUNGU_NM', 'avg_service_per_capita']])

## 3. 버퍼 거리별 평균 접근성 점수 변화 분석

버퍼 거리(100m, 300m, 500m, 1000m)에 따른 평균 접근성 점수 변화를 시각화합니다.

In [ ]:
# 전체 공원 유형에 대한 버퍼 거리별 평균 점수 계산
df_all_parks = df_multi[df_multi['park_types'] == '전체'].copy()

buffer_stats = df_all_parks.groupby('buffer_distance').agg({
    'avg_service_per_capita': 'mean',
    'coverage_ratio': 'mean',
    'total_population': 'sum'
}).reset_index()

buffer_stats = buffer_stats.sort_values('buffer_distance')

# 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 왼쪽: 평균 접근성 점수
ax1.plot(buffer_stats['buffer_distance'], buffer_stats['avg_service_per_capita'], 
         marker='o', linewidth=2.5, markersize=10, color='#1f77b4', label='평균 서비스 점수')
ax1.fill_between(buffer_stats['buffer_distance'], buffer_stats['avg_service_per_capita'], 
                  alpha=0.3, color='#1f77b4')
ax1.axhline(y=9, color='red', linestyle='--', linewidth=2, label='WHO 권장 기준 (9 m²/인)')

ax1.set_xlabel('버퍼 거리 (m)', fontsize=12, fontweight='bold')
ax1.set_ylabel('평균 서비스 점수 (m²/인)', fontsize=12, fontweight='bold')
ax1.set_title('버퍼 거리별 평균 공원 접근성 점수', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)

# 값 레이블
for idx, row in buffer_stats.iterrows():
    ax1.text(row['buffer_distance'], row['avg_service_per_capita'] + 0.3, 
             f"{row['avg_service_per_capita']:.2f}",
             ha='center', fontsize=10, fontweight='bold')

# 오른쪽: 커버리지 비율
ax2.plot(buffer_stats['buffer_distance'], buffer_stats['coverage_ratio'], 
         marker='s', linewidth=2.5, markersize=10, color='#ff7f0e', label='평균 커버리지')
ax2.fill_between(buffer_stats['buffer_distance'], buffer_stats['coverage_ratio'], 
                  alpha=0.3, color='#ff7f0e')

ax2.set_xlabel('버퍼 거리 (m)', fontsize=12, fontweight='bold')
ax2.set_ylabel('커버리지 비율 (%)', fontsize=12, fontweight='bold')
ax2.set_title('버퍼 거리별 평균 커버리지 비율', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=10)

# 값 레이블
for idx, row in buffer_stats.iterrows():
    ax2.text(row['buffer_distance'], row['coverage_ratio'] + 1, 
             f"{row['coverage_ratio']:.1f}%",
             ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

# 통계 요약
print("버퍼 거리별 통계:")
print(buffer_stats.to_string(index=False))

## 4. 버퍼 거리별 Gini 계수 변화 분석

버퍼 거리에 따른 지역 간 불평등도(Gini 계수) 변화를 분석합니다.

In [ ]:
# 버퍼 거리별 Gini 계수 계산
gini_by_buffer = []

for buffer_dist in sorted(df_all_parks['buffer_distance'].unique()):
    df_buffer = df_all_parks[df_all_parks['buffer_distance'] == buffer_dist]
    gini = calculate_gini(df_buffer['avg_service_per_capita'].values)
    gini_by_buffer.append({
        'buffer_distance': buffer_dist,
        'gini_coefficient': gini
    })

df_gini = pd.DataFrame(gini_by_buffer)

# 시각화
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(df_gini['buffer_distance'], df_gini['gini_coefficient'], 
        marker='D', linewidth=3, markersize=12, color='#d62728', label='Gini 계수')
ax.fill_between(df_gini['buffer_distance'], df_gini['gini_coefficient'], 
                 alpha=0.2, color='#d62728')

ax.set_xlabel('버퍼 거리 (m)', fontsize=12, fontweight='bold')
ax.set_ylabel('Gini 계수', fontsize=12, fontweight='bold')
ax.set_title('버퍼 거리별 지역 간 공원 접근성 불평등도 (Gini 계수)', 
             fontsize=14, fontweight='bold', pad=20)

# Gini 해석 가이드 추가
ax.axhline(y=0.4, color='green', linestyle=':', linewidth=1.5, alpha=0.5, label='양호 기준 (0.4)')
ax.axhline(y=0.6, color='orange', linestyle=':', linewidth=1.5, alpha=0.5, label='경계 기준 (0.6)')

ax.grid(True, alpha=0.3)
ax.legend(fontsize=10, loc='best')

# 값 레이블
for idx, row in df_gini.iterrows():
    ax.text(row['buffer_distance'], row['gini_coefficient'] + 0.01, 
            f"{row['gini_coefficient']:.4f}",
            ha='center', fontsize=10, fontweight='bold')

# Y축 범위 설정
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

# Gini 계수 해석
print("\\n버퍼 거리별 Gini 계수:")
print(df_gini.to_string(index=False))
print("\\nGini 계수 해석:")
print("  0.0 ~ 0.3: 평등한 분포")
print("  0.3 ~ 0.4: 비교적 평등")
print("  0.4 ~ 0.6: 중간 수준의 불평등")
print("  0.6 ~ 0.8: 높은 불평등")
print("  0.8 ~ 1.0: 매우 높은 불평등")

## 5. 공원 유형별 접근성 기여도 분석

공원 유형별 개수 비율과 접근성 기여도를 비교 분석합니다.

In [ ]:
gdf_park

In [ ]:
# 500m 버퍼 기준 공원 유형별 데이터 추출
df_500m_bytype = df_multi[df_multi['buffer_distance'] == 500].copy()

# '전체' 제외하고 개별 유형만 필터링
df_500m_bytype = df_500m_bytype[df_500m_bytype['park_types'] != '전체']

# 공원 유형별 평균 서비스 점수 계산
parktype_stats = df_500m_bytype.groupby('park_types').agg({
    'avg_service_per_capita': 'mean'
}).reset_index()

# 공원 유형별 개수 계산
park_counts = gdf_park['LABEL'].value_counts().reset_index()
park_counts.columns = ['park_types', 'park_count']
park_counts['count_ratio'] = (park_counts['park_count'] / park_counts['park_count'].sum()) * 100

# 병합
parktype_stats = parktype_stats.merge(park_counts, on='park_types', how='left')

# 접근성 기여도 비율 계산
parktype_stats['contribution_ratio'] = (
    parktype_stats['avg_service_per_capita'] / 
    parktype_stats['avg_service_per_capita'].sum() * 100
)

parktype_stats = parktype_stats.sort_values('avg_service_per_capita', ascending=False)

# 시각화
fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(parktype_stats))
width = 0.35

bars1 = ax.bar(x - width/2, parktype_stats['count_ratio'], width, 
               label='공원 수 비율 (%)', color='#1f77b4', alpha=0.7)
bars2 = ax.bar(x + width/2, parktype_stats['contribution_ratio'], width, 
               label='접근성 기여도 (%)', color='#ff7f0e', alpha=0.7)

ax.set_xlabel('공원 유형', fontsize=12, fontweight='bold')
ax.set_ylabel('비율 (%)', fontsize=12, fontweight='bold')
ax.set_title('공원 유형별 개수 비율 vs 접근성 기여도', fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(parktype_stats['park_types'], rotation=45, ha='right', fontsize=10)
ax.legend(fontsize=11, loc='upper right')
ax.grid(axis='y', alpha=0.3)

# 값 레이블
for i, (idx, row) in enumerate(parktype_stats.iterrows()):
    ax.text(i - width/2, row['count_ratio'] + 1, f"{row['count_ratio']:.1f}%", 
            ha='center', va='bottom', fontsize=9)
    ax.text(i + width/2, row['contribution_ratio'] + 1, f"{row['contribution_ratio']:.1f}%", 
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# 통계 요약
print("\\n공원 유형별 통계:")
print(parktype_stats[['park_types', 'park_count', 'count_ratio', 
                       'avg_service_per_capita', 'contribution_ratio']].to_string(index=False))

## 5-1. 공원 유형별 데이터 확인 및 NaN 문제 해결

공원 데이터의 유형 분포를 확인하고 NaN 값의 원인을 파악합니다.

## 6. 공원 유형별 접근성 기여 비중 (파이차트)

공원 유형별 접근성 기여 비중을 파이차트로 시각화합니다.

In [ ]:
# 기여도 기준 내림차순 정렬
parktype_sorted = parktype_stats.sort_values(
    by='contribution_ratio',
    ascending=False
)

# 상위 5개
top5 = parktype_sorted.head(5)

# 나머지 -> 기타
others = parktype_sorted.iloc[5:]

others_row = pd.DataFrame({
    'park_types': ['기타'],
    'contribution_ratio': [others['contribution_ratio'].sum()]
})

# 파이차트용 데이터
pie_df = pd.concat([top5[['park_types', 'contribution_ratio']], others_row],
                    ignore_index=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

colors = plt.cm.Set3(np.linspace(0, 1, len(pie_df)))

# 왼쪽: 공원 수 비율
wedges1, texts1, autotexts1 = ax1.pie(
    pie_df['contribution_ratio'],
    labels=pie_df['park_types'],
    autopct='%1.1f%%',
    startangle=90,
    colors=colors,
    textprops={'fontsize': 10}
)

ax1.set_title('공원 유형별 개수 비율 (상위 5개)', fontsize=14, fontweight='bold', pad=20)

# 오른쪽: 접근성 기여도
wedges2, texts2, autotexts2 = ax2.pie(
    pie_df['contribution_ratio'],
    labels=pie_df['park_types'],
    autopct='%1.1f%%',
    startangle=90,
    colors=colors,
    textprops={'fontsize': 10}
)

ax2.set_title('공원 유형별 접근성 기여도 (상위 5개)', fontsize=14, fontweight='bold', pad=20)

# 퍼센트 텍스트 스타일
for autotext in autotexts1 + autotexts2:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(9)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# 고급스러운 muted 컬러 팔레트
colors = [
    "#5ab4ac",  # 메인 (도시자연공원)
    "#b2abd2",
    "#80b1d3",
    "#fccde5",
    "#bc80bd",
    "#fdb462"   # 기타
]

def draw_donut(ax, data, labels, title):
    wedges, texts, autotexts = ax.pie(
        data,
        labels=labels,
        autopct='%1.1f%%',
        startangle=90,
        colors=colors,
        pctdistance=0.75,
        labeldistance=1.1,
        wedgeprops=dict(width=0.4, edgecolor='white')
    )

    # 퍼센트 텍스트 스타일
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(10)
        autotext.set_fontweight('bold')

    # 중앙 원
    centre_circle = plt.Circle((0, 0), 0.55, fc='white')
    ax.add_artist(centre_circle)

    ax.set_title(title, fontsize=15, fontweight='bold', pad=25)
    ax.axis('equal')


# 왼쪽: 공원 수 비율
draw_donut(
    axes[0],
    pie_df['contribution_ratio'],
    pie_df['park_types'],
    '공원 유형별 개수 비율 (상위 5개)'
)

# 오른쪽: 접근성 기여도
draw_donut(
    axes[1],
    pie_df['contribution_ratio'],
    pie_df['park_types'],
    '공원 유형별 접근성 기여도 (상위 5개)'
)

plt.tight_layout()
plt.show()


## 7. 행정동별 접근성 상·하위 10개 지역 비교

접근성이 가장 높은 상위 10개 행정동과 가장 낮은 하위 10개 행정동을 비교합니다.

In [ ]:
# 상위/하위 10개 행정동 추출
top10 = df_500m.nlargest(10, 'avg_service_per_capita').copy()
bottom10 = df_500m.nsmallest(10, 'avg_service_per_capita').copy()

top10['rank'] = '상위 10개'
bottom10['rank'] = '하위 10개'

# 결합
comparison = pd.concat([top10, bottom10])

# 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 10))

# 상위 10개
top10_sorted = top10.sort_values('avg_service_per_capita')
bars1 = ax1.barh(
    range(len(top10_sorted)),
    top10_sorted['avg_service_per_capita'],
    color='#2ca02c',
    alpha=0.7
)

ax1.set_yticks(range(len(top10_sorted)))
ax1.set_yticklabels([f"{row['ADM_NM']}\\n({row['SIGUNGU_NM']})" 
                      for _, row in top10_sorted.iterrows()],
                     fontsize=9)
ax1.set_xlabel('공원 접근성 점수 (m²/인)', fontsize=11, fontweight='bold')
ax1.set_title('접근성 상위 10개 행정동', fontsize=14, fontweight='bold', pad=15)
ax1.axvline(x=9, color='red', linestyle='--', linewidth=2, label='WHO 기준 (9 m²/인)')
ax1.axvline(x=seoul_avg, color='blue', linestyle=':', linewidth=2, 
            label=f'서울시 평균 ({seoul_avg:.2f} m²/인)')
ax1.legend(fontsize=9, loc='lower right')
ax1.grid(axis='x', alpha=0.3)

# 값 레이블
for i, (idx, row) in enumerate(top10_sorted.iterrows()):
    ax1.text(row['avg_service_per_capita'] + 0.5, i, 
             f"{row['avg_service_per_capita']:.2f}",
             va='center', fontsize=8)

# 하위 10개
bottom10_sorted = bottom10.sort_values('avg_service_per_capita', ascending=False)
bars2 = ax2.barh(
    range(len(bottom10_sorted)),
    bottom10_sorted['avg_service_per_capita'],
    color='#d62728',
    alpha=0.7
)

ax2.set_yticks(range(len(bottom10_sorted)))
ax2.set_yticklabels([f"{row['ADM_NM']}\\n({row['SIGUNGU_NM']})" 
                      for _, row in bottom10_sorted.iterrows()],
                     fontsize=9)
ax2.set_xlabel('공원 접근성 점수 (m²/인)', fontsize=11, fontweight='bold')
ax2.set_title('접근성 하위 10개 행정동', fontsize=14, fontweight='bold', pad=15)
ax2.axvline(x=9, color='red', linestyle='--', linewidth=2, label='WHO 기준 (9 m²/인)')
ax2.axvline(x=seoul_avg, color='blue', linestyle=':', linewidth=2, 
            label=f'서울시 평균 ({seoul_avg:.2f} m²/인)')
ax2.legend(fontsize=9, loc='lower right')
ax2.grid(axis='x', alpha=0.3)

# 값 레이블
for i, (idx, row) in enumerate(bottom10_sorted.iterrows()):
    ax2.text(row['avg_service_per_capita'] + 0.1, i, 
             f"{row['avg_service_per_capita']:.2f}",
             va='center', fontsize=8)

plt.tight_layout()
plt.show()

# 통계 요약
print("\\n상위 10개 행정동:")
print(top10[['ADM_NM', 'SIGUNGU_NM', 'avg_service_per_capita']].to_string(index=False))
print("\\n하위 10개 행정동:")
print(bottom10[['ADM_NM', 'SIGUNGU_NM', 'avg_service_per_capita']].to_string(index=False))

# 격차 분석
gap = top10['avg_service_per_capita'].min() - bottom10['avg_service_per_capita'].max()
print(f"\\n상위 최소값과 하위 최대값의 격차: {gap:.2f} m²/인")